# TSA Chapter 12: Long Memory Processes and Fractional Integration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/TSA/blob/main/TSA_ch12/TSA_ch12_long_memory/TSA_ch12_long_memory.ipynb)

In [ ]:
!pip install matplotlib numpy scipy statsmodels pywt -q

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import periodogram

In [ ]:
# Style configuration
COLORS = {
    'blue':   '#1A3A6E',
    'red':    '#DC3545',
    'green':  '#2E7D32',
    'orange': '#E67E22',
    'gray':   '#666666',
    'purple': '#8E44AD',
}

plt.rcParams.update({
    'axes.facecolor':     'none',
    'figure.facecolor':   'none',
    'savefig.transparent': True,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          False,
    'font.size':          9,
    'axes.labelsize':     9,
    'axes.titlesize':     10,
    'legend.fontsize':    8,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
})

In [ ]:
# Chart: ch12_estimation_comparison
# Long memory ARFIMA(0,d,0): ACF and log-log periodogram slope
def simulate_arfima(d, N, seed=0):
    rng = np.random.default_rng(seed)
    eps = rng.normal(0, 1, N + 500)
    max_lag = min(N, 500)
    psi = np.zeros(max_lag)
    psi[0] = 1.0
    for j in range(1, max_lag):
        psi[j] = psi[j-1] * (d + j - 1) / j
    x = np.convolve(eps, psi)[:N + 500]
    return x[500:]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
d_vals = [0.0, 0.3, 0.45]
cols   = [COLORS['gray'], COLORS['blue'], COLORS['red']]
N = 1024

for d, col in zip(d_vals, cols):
    x = simulate_arfima(d, N)
    fr, psd = periodogram(x)
    fr_pos = fr[1:N//4]
    psd_pos = psd[1:N//4]
    axes[1].loglog(fr_pos, psd_pos, color=col, lw=1, alpha=0.8, label=f'd={d}')
    if d > 0:
        slope_line = psd_pos[5] * (fr_pos / fr_pos[5])**(-2*d)
        axes[1].loglog(fr_pos, slope_line, color=col, lw=1.5, linestyle='--', alpha=0.5)
    acf = np.array([np.corrcoef(x[:-h], x[h:])[0,1] if h > 0 else 1.0 for h in range(50)])
    axes[0].plot(acf, color=col, lw=1.8, label=f'd={d}')

axes[0].set_title('ACF of ARFIMA(0,d,0)')
axes[0].set_xlabel('Lag h')
axes[0].set_ylabel('Autocorrelation')
axes[1].set_title('Log-Log Periodogram (slope = −2d)')
axes[1].set_xlabel('log Frequency')
axes[1].set_ylabel('log PSD')

for ax in axes:
    ax.set_facecolor('none')
    ax.spines[['top','right']].set_visible(False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.savefig('ch12_estimation_comparison.pdf', bbox_inches='tight', dpi=150)
plt.savefig('ch12_estimation_comparison.png', bbox_inches='tight', dpi=150)
plt.show()